# 07: Extract and preprocess the fresh confirmation set

`data/eyepacs/raw/train_combined.zip` (35 GB) is five zip wrappers concatenated end to end,
each holding one part of Kaggle's split archive (`train.zip.001` ... `.005`). This notebook:

1. Walks the file wrapper by wrapper, extracts each part, **verifies its CRC**, and joins the
   parts into the real `train.zip` on local disk.
2. Confirms `train.zip` holds all 35,126 labelled images.
3. **Fidelity gate:** re-runs the recovered v1 preprocessing on ~40 images the models were
   trained on and compares pixel by pixel with the processed batches. Stops if they differ.
4. Preprocesses the 15,794 fresh images (7,897 patients never used anywhere) with the same
   pipeline, zips them, and saves to `dr_ensemble_v2/06_fresh_images/`.
5. Writes `01_splits/eyepacs_split_fresh_v2.csv` and a provenance record to `99_audit`.

**No model is run and nothing is evaluated.** This only prepares images.

Needs ~45 GB free local disk. Use a runtime with many CPU cores (the A100 runtime has 12):
preprocessing is CPU-bound and takes roughly 10 minutes on 12 cores, over an hour on 2.
Expect 30-45 minutes total. Run cells 1-6.

In [1]:
# Cell 1: mount, paths, disk check, fresh-set definition
from google.colab import drive
drive.mount('/content/drive')
import os, io, json, struct, zlib, zipfile, shutil, time, hashlib, datetime
import numpy as np, pandas as pd, PIL
from PIL import Image

BASE    = '/content/drive/MyDrive/stochastic_orthogonality/data'
SRC     = f'{BASE}/eyepacs/raw/train_combined.zip'
BATCHES = [f'{BASE}/eyepacs_full/eyepacs_processed_batch{i}.zip' for i in range(1, 6)]
LABELS  = f'{BASE}/eyepacs_full/trainLabels.csv'
V2      = f'{BASE}/dr_ensemble_v2'
OUTDIR  = f'{V2}/06_fresh_images'; os.makedirs(OUTDIR, exist_ok=True)
LOCAL_ZIP = '/content/train.zip'
PNG_DIR   = '/content/fresh_png'; os.makedirs(PNG_DIR, exist_ok=True)

free = shutil.disk_usage('/content').free / 1e9
print(f'free local disk: {free:.0f} GB | CPU cores: {os.cpu_count()} | PIL {PIL.__version__}')
assert free > 45, 'need ~45 GB free local disk'

labels = pd.read_csv(LABELS); labels.columns = ['image_id', 'level']; labels['image_id'] = labels['image_id'].astype(str)
used = pd.concat([pd.read_csv(f'{V2}/01_splits/eyepacs_split_{s}_v2.csv') for s in ['train', 'val', 'test']])
used_ids = set(used['image_id'].astype(str))
fresh = labels[~labels['image_id'].isin(used_ids)].copy()
fresh['patient_id'] = fresh['image_id'].str.split('_').str[0]
fresh['eye'] = fresh['image_id'].str.split('_').str[-1]
fresh['binary_label'] = (fresh['level'] >= 2).astype(int)
assert not (set(fresh['patient_id']) & set(used['patient_id'].astype(str))), 'patient overlap'
print(f'fresh set: {len(fresh)} images, {fresh.patient_id.nunique()} patients (patient-disjoint from v2 splits)')

Mounted at /content/drive
free local disk: 221 GB | CPU cores: 2 | PIL 11.3.0
fresh set: 15794 images, 7897 patients (patient-disjoint from v2 splits)


In [2]:
# Cell 2: unwrap the five concatenated wrappers into one train.zip, verifying each part's CRC
def parse_local(f, off):
    f.seek(off); h = f.read(30)
    if h[:4] != b'PK\x03\x04': return None
    ver, flag, method, _, _, crc, csize, usize, nlen, elen = struct.unpack('<HHHHHIIIHH', h[4:30])
    name = f.read(nlen).decode(); extra = f.read(elen)
    i = 0
    while i + 4 <= len(extra):                                   # zip64 sizes (parts exceed 4 GB)
        hid, hsz = struct.unpack('<HH', extra[i:i + 4]); body = extra[i + 4:i + 4 + hsz]; j = 0
        if hid == 1:
            if usize == 0xFFFFFFFF: usize = struct.unpack('<Q', body[j:j + 8])[0]; j += 8
            if csize == 0xFFFFFFFF: csize = struct.unpack('<Q', body[j:j + 8])[0]; j += 8
        i += 4 + hsz
    return {'name': name, 'flag': flag, 'method': method, 'crc': crc, 'csize': csize, 'usize': usize,
            'data_off': off + 30 + nlen + elen}

CHUNK = 64 << 20
def copy_member(f, out, m):
    """Stream one member's payload into out; returns (crc32, compressed bytes consumed)."""
    f.seek(m['data_off']); crc = 0
    if m['method'] == 0:                                          # stored
        left = m['csize']
        while left:
            b = f.read(min(CHUNK, left)); out.write(b); crc = zlib.crc32(b, crc); left -= len(b)
        return crc, m['csize']
    if m['method'] == 8:                                          # deflate
        d = zlib.decompressobj(-15); used_ = 0
        while not d.eof:
            b = f.read(CHUNK)
            if not b: raise IOError('truncated deflate stream')
            used_ += len(b); o = d.decompress(b); out.write(o); crc = zlib.crc32(o, crc)
        return crc, used_ - len(d.unused_data)
    raise ValueError(f"unsupported compression method {m['method']}")

size = os.path.getsize(SRC); off = 0; parts = []; t0 = time.time()
with open(SRC, 'rb') as f, open(LOCAL_ZIP, 'wb') as out:
    while off < size:
        m = parse_local(f, off)
        if m is None: raise IOError(f'no local header at offset {off}')
        print(f"  {m['name']}: method {m['method']}, {m['usize']/1e9:.2f} GB ... ", end='', flush=True)
        crc, consumed = copy_member(f, out, m)
        ok = (crc & 0xFFFFFFFF) == m['crc'] or m['flag'] & 8
        parts.append({'name': m['name'], 'bytes': m['usize'], 'crc_ok': bool((crc & 0xFFFFFFFF) == m['crc'])})
        print(f"CRC {'OK' if parts[-1]['crc_ok'] else 'MISMATCH'} ({time.time()-t0:.0f}s)")
        if not ok: raise IOError(f"CRC mismatch in {m['name']}")
        f.seek(m['data_off'] + consumed); tail = f.read(1 << 20)   # skip this wrapper's index
        nxt = tail.find(b'PK\x03\x04')
        if nxt < 0: break
        off = m['data_off'] + consumed + nxt

print('parts:', [p['name'] for p in parts])
assert [p['name'] for p in parts] == [f'train.zip.00{i}' for i in range(1, 6)], 'unexpected parts'
print(f'assembled train.zip: {os.path.getsize(LOCAL_ZIP)/1e9:.2f} GB')

  train.zip.001: method 8, 8.39 GB ... CRC OK (209s)
  train.zip.002: method 8, 8.39 GB ... CRC OK (442s)
  train.zip.003: method 8, 8.39 GB ... CRC OK (678s)
  train.zip.004: method 8, 8.39 GB ... CRC OK (900s)
  train.zip.005: method 8, 1.43 GB ... CRC OK (931s)
parts: ['train.zip.001', 'train.zip.002', 'train.zip.003', 'train.zip.004', 'train.zip.005']
assembled train.zip: 34.99 GB


In [3]:
# Cell 3: confirm the assembled archive holds every labelled image
with open(LOCAL_ZIP, 'rb') as f: sig = f.read(4)
print('assembled archive signature:', sig, '(PK\\x03\\x04 = ordinary zip)')
Z = zipfile.ZipFile(LOCAL_ZIP)
IDX = {os.path.splitext(os.path.basename(n))[0]: n for n in Z.namelist() if n.lower().endswith(('.jpeg', '.jpg', '.png'))}
missing_all = set(labels['image_id']) - set(IDX)
missing_fresh = set(fresh['image_id']) - set(IDX)
print(f'images in train.zip: {len(IDX)} | labelled missing: {len(missing_all)} | fresh missing: {len(missing_fresh)}')
assert not missing_fresh, 'fresh images missing from the assembled archive'

assembled archive signature: b'PK\x03\x04' (PK\x03\x04 = ordinary zip)
images in train.zip: 35126 | labelled missing: 0 | fresh missing: 0


In [4]:
# Cell 4: FIDELITY GATE: recovered v1 pipeline vs the processed images the models trained on
def v1_preprocess(fp):
    """Recovered from v1 notebook1_data_setup: RGB, resize 512x512 Lanczos (aspect not preserved)."""
    return Image.open(fp).convert('RGB').resize((512, 512), Image.LANCZOS)

PROC = {}
for b in BATCHES:
    with zipfile.ZipFile(b) as zb:
        for n in zb.namelist():
            if not n.endswith('/'): PROC[os.path.splitext(os.path.basename(n))[0]] = (b, n)
used_df = used[['image_id', 'level']].copy(); used_df['image_id'] = used_df['image_id'].astype(str)
sample = pd.concat([g.sample(8, random_state=20260922 + int(k)) for k, g in used_df.groupby('level')])
zb = {b: zipfile.ZipFile(b) for b in BATCHES}
rows = []
for iid in sample['image_id']:
    rep = np.asarray(v1_preprocess(io.BytesIO(Z.read(IDX[iid]))), dtype=np.int16)
    b, n = PROC[iid]; proc = Image.open(io.BytesIO(zb[b].read(n)))
    pa = np.asarray(proc.convert('RGB'), dtype=np.int16)
    d = np.abs(pa - rep) if pa.shape == rep.shape else None
    rows.append({'image_id': iid, 'proc_size': proc.size, 'proc_format': proc.format,
                 'same_shape': pa.shape == rep.shape,
                 'max_abs_diff': None if d is None else int(d.max()),
                 'mean_abs_diff': None if d is None else round(float(d.mean()), 4)})
FID = pd.DataFrame(rows)
print(FID.to_string(index=False))
FIDELITY_OK = bool(FID['same_shape'].all() and FID['mean_abs_diff'].max() < 1.0)
print(f"\ncompared {len(FID)} | exact matches {int((FID['max_abs_diff'] == 0).sum())} | "
      f"worst mean diff {FID['mean_abs_diff'].max()} | worst max diff {FID['max_abs_diff'].max()} | "
      f"GATE {'PASSED' if FIDELITY_OK else 'FAILED'}")
if not FIDELITY_OK:
    raise SystemExit('Preprocessing does not reproduce v1. Stop and report; do not preprocess the fresh set.')

   image_id  proc_size proc_format  same_shape  max_abs_diff  mean_abs_diff
19506_right (512, 512)         PNG        True             0            0.0
 19008_left (512, 512)         PNG        True             0            0.0
 13101_left (512, 512)         PNG        True             0            0.0
 17097_left (512, 512)         PNG        True             0            0.0
22134_right (512, 512)         PNG        True             0            0.0
25956_right (512, 512)         PNG        True             0            0.0
28174_right (512, 512)         PNG        True             0            0.0
13177_right (512, 512)         PNG        True             0            0.0
20361_right (512, 512)         PNG        True             0            0.0
 19986_left (512, 512)         PNG        True             0            0.0
19004_right (512, 512)         PNG        True             0            0.0
12268_right (512, 512)         PNG        True             0            0.0
14614_right 

In [5]:
# Cell 5: preprocess the 15,794 fresh images in parallel, zip, save to Drive
from multiprocessing import Pool
_Z = None
def _init():
    global _Z; _Z = zipfile.ZipFile(LOCAL_ZIP)
def _work(args):
    iid, member = args
    out = f'{PNG_DIR}/{iid}.png'
    if os.path.exists(out) and os.path.getsize(out) > 0: return iid, 'skip'
    try:
        v1_preprocess(io.BytesIO(_Z.read(member))).save(out); return iid, 'ok'
    except Exception as e:
        return iid, f'error: {e}'

jobs = [(i, IDX[i]) for i in fresh['image_id']]
t0 = time.time(); results = []
with Pool(os.cpu_count(), initializer=_init) as pool:
    for k, r in enumerate(pool.imap_unordered(_work, jobs, chunksize=16), 1):
        results.append(r)
        if k % 2000 == 0: print(f'  {k}/{len(jobs)} ({time.time()-t0:.0f}s)')
errors = [r for r in results if r[1].startswith('error')]
print(f'processed {len(results)} in {time.time()-t0:.0f}s | errors: {len(errors)}', errors[:5])
assert not errors

# zip locally in four batches, then copy to Drive (many small files write slowly to Drive)
ids = sorted(fresh['image_id']); ZIPS = []
for bi, chunk in enumerate(np.array_split(ids, 4), 1):
    zp = f'/content/eyepacs_fresh_batch{bi}_v2.zip'
    with zipfile.ZipFile(zp, 'w', zipfile.ZIP_STORED) as zw:
        for iid in chunk: zw.write(f'{PNG_DIR}/{iid}.png', f'{iid}.png')
    shutil.copy(zp, f'{OUTDIR}/{os.path.basename(zp)}'); ZIPS.append(os.path.basename(zp))
    print(f'  saved {os.path.basename(zp)} ({os.path.getsize(zp)/1e9:.2f} GB)')

  2000/15794 (651s)
  4000/15794 (1321s)
  6000/15794 (2004s)
  8000/15794 (2666s)
  10000/15794 (3334s)
  12000/15794 (3995s)
  14000/15794 (4650s)
processed 15794 in 5232s | errors: 0 []
  saved eyepacs_fresh_batch1_v2.zip (0.93 GB)
  saved eyepacs_fresh_batch2_v2.zip (0.92 GB)
  saved eyepacs_fresh_batch3_v2.zip (0.92 GB)
  saved eyepacs_fresh_batch4_v2.zip (0.93 GB)


In [6]:
# Cell 6: fresh split file + provenance record
cols = ['image_id', 'patient_id', 'eye', 'level', 'binary_label']
fresh[cols].sort_values('image_id').to_csv(f'{V2}/01_splits/eyepacs_split_fresh_v2.csv', index=False)
REC = {
  'generated_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(), 'pil_version': PIL.__version__,
  'source': 'data/eyepacs/raw/train_combined.zip: five concatenated zip wrappers around Kaggle train.zip.001-.005',
  'parts': parts, 'assembled_images': len(IDX),
  'fresh_set': {'images': int(len(fresh)), 'patients': int(fresh.patient_id.nunique()),
                'patient_overlap_with_v2_splits': 0,
                'why_unused': 'v1 restricted labels to images present in its processed folder; these were never processed'},
  'preprocessing': 'recovered v1 pipeline: RGB, resize 512x512 Lanczos, PNG',
  'fidelity_gate': {'passed': FIDELITY_OK, 'compared': int(len(FID)),
                    'worst_mean_abs_diff': float(FID['mean_abs_diff'].max()),
                    'worst_max_abs_diff': int(FID['max_abs_diff'].max())},
  'outputs': {'images': [f'06_fresh_images/{z}' for z in ZIPS], 'split': '01_splits/eyepacs_split_fresh_v2.csv'},
  'note': 'Preparation only. No model run, no evaluation.',
}
with open(f'{V2}/99_audit/fresh_set_preparation_v2.json', 'w') as f: json.dump(REC, f, indent=2, default=str)
print(json.dumps({k: REC[k] for k in ['parts', 'assembled_images', 'fidelity_gate', 'outputs']}, indent=2, default=str))

{
  "parts": [
    {
      "name": "train.zip.001",
      "bytes": 8388608000,
      "crc_ok": true
    },
    {
      "name": "train.zip.002",
      "bytes": 8388608000,
      "crc_ok": true
    },
    {
      "name": "train.zip.003",
      "bytes": 8388608000,
      "crc_ok": true
    },
    {
      "name": "train.zip.004",
      "bytes": 8388608000,
      "crc_ok": true
    },
    {
      "name": "train.zip.005",
      "bytes": 1434013506,
      "crc_ok": true
    }
  ],
  "assembled_images": 35126,
  "fidelity_gate": {
    "passed": true,
    "compared": 40,
    "worst_mean_abs_diff": 0.0,
    "worst_max_abs_diff": 0
  },
  "outputs": {
    "images": [
      "06_fresh_images/eyepacs_fresh_batch1_v2.zip",
      "06_fresh_images/eyepacs_fresh_batch2_v2.zip",
      "06_fresh_images/eyepacs_fresh_batch3_v2.zip",
      "06_fresh_images/eyepacs_fresh_batch4_v2.zip"
    ],
    "split": "01_splits/eyepacs_split_fresh_v2.csv"
  }
}
